In [10]:
import os, sys
from pathlib import Path

# Notebooks have no __file__; walk up from cwd to the repo root.
PROJECT_ROOT = "/Users/soniaatre/Cornell2025/Research/PublicView/analyze-sycophancy-public"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, "src")
import collections
import json
import re

import numpy as np
import pandas as pd

from data_preprocessing.lexicon import ANY_RE

In [11]:
POSITIVE = "data/text_comments_sycophan_for_embed.jsonl"
BACKGROUND = "data/text_comments_for_embed_2022-12-22_2026-9-18.jsonl"
EVERY = 20              # background = every k-th comment
MIN_POS_COUNT = 20
TOP = 300
OUTPUT_FILE = "data/candidates/logodds_terms.csv"

In [12]:
TOKEN_RE = re.compile(r"[a-z][a-z']+")

STOP = set("""the a an and or but if of to in on at for with from by as is are was were be been being it its it's this that
these those i me my we our you your he she they them their his her im i'm ive i've dont don't do does did doesn't didn't not no
so just have has had can could would should will not than then there here what which who how when where why all any some very
more most much really also about into out up down over just like get got one thing things because too""".split())


def ngrams(text: str):
    toks = [t for t in TOKEN_RE.findall(text.lower())]
    uni = [t for t in toks if t not in STOP]
    bi = [f"{a} {b}" for a, b in zip(toks, toks[1:]) if a not in STOP and b not in STOP]
    return uni + bi

In [13]:
ngrams("This is such a great and insightful point, you're absolutely right!")

['such',
 'great',
 'insightful',
 'point',
 "you're",
 'absolutely',
 'right',
 'such great',
 'insightful point',
 "point you're",
 "you're absolutely",
 'absolutely right']

In [14]:
BOT_RE = re.compile(r"I am a bot, and this action was performed automatically|^Hey /u/", re.I)


def count_file(path: str, every: int = 1) -> tuple[collections.Counter, int]:
    counts, n_docs = collections.Counter(), 0
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i % every:
                continue
            text = json.loads(line)["text"]
            if BOT_RE.search(text):
                continue
            counts.update(set(ngrams(text)))
            n_docs += 1
    return counts, n_docs

In [15]:
pos, n_pos = count_file(POSITIVE)
print(f"positive: {n_pos} docs, {len(pos)} distinct terms")
pos.most_common(20)

positive: 18762 docs, 217669 distinct terms


[('people', 3482),
 ('ai', 2963),
 ('sycophantic', 2924),
 ('chatgpt', 2662),
 ('sycophancy', 2400),
 ('think', 2340),
 ('even', 2116),
 ('use', 2107),
 ('want', 2012),
 ('gpt', 1920),
 ('model', 1876),
 ('way', 1834),
 ('know', 1735),
 ('now', 1733),
 ('something', 1729),
 ('back', 1517),
 ('good', 1487),
 ('time', 1470),
 ('say', 1465),
 ('only', 1452)]

In [16]:
bg, n_bg = count_file(BACKGROUND, EVERY)
print(f"background: {n_bg} docs, {len(bg)} distinct terms")
bg.most_common(20)

background: 202130 docs, 1321795 distinct terms


[('ai', 30857),
 ('people', 23573),
 ('chatgpt', 23165),
 ('use', 19237),
 ('think', 18883),
 ('even', 17095),
 ('know', 16058),
 ('now', 14940),
 ('time', 14763),
 ('make', 14339),
 ('way', 13826),
 ('gpt', 13804),
 ('only', 13398),
 ('something', 13262),
 ('https', 12239),
 ('want', 11893),
 ('see', 11887),
 ('same', 11878),
 ('good', 11735),
 ('other', 11547)]

In [20]:
pos_filtered = collections.Counter({t: c for t, c in pos.items() if c >= MIN_POS_COUNT})
print(f"{len(pos_filtered)} / {len(pos)} terms survive the min_pos_count={MIN_POS_COUNT} filter")

4151 / 217669 terms survive the min_pos_count=20 filter


In [21]:
def log_odds(pos: collections.Counter, bg: collections.Counter, alpha0: float = 500.0) -> pd.DataFrame:
    terms = [t for t in pos if bg.get(t, 0) or pos[t]]
    y_i = np.array([pos[t] for t in terms], dtype=float)
    y_j = np.array([bg.get(t, 0) for t in terms], dtype=float)
    n_i, n_j = y_i.sum(), sum(bg.values())
    prior = (y_i + y_j) / (n_i + n_j) * alpha0
    a0 = prior.sum()
    delta = np.log((y_i + prior) / (n_i + a0 - y_i - prior)) - np.log((y_j + prior) / (n_j + a0 - y_j - prior))
    z = delta / np.sqrt(1 / (y_i + prior) + 1 / (y_j + prior))
    return pd.DataFrame(dict(term=terms, pos_docs=y_i.astype(int), bg_docs=y_j.astype(int), z=z))

In [22]:
scored = log_odds(pos_filtered, bg)
print(f"{len(scored)} terms scored")
scored.sort_values("z", ascending=False).head(20)

4151 terms scored


,term,pos_docs,bg_docs,z
122,sycophantic,2924,141,66.050535
622,sycophancy,2400,126,61.365854
577,model,1876,7815,47.858568
171,personality,811,1446,47.355719
177,user,1134,3379,45.586961
298,sycophant,1337,61,43.896249
452,behavior,673,1209,43.035403
321,people,3482,23573,41.097089
513,users,902,3031,38.068171
1567,instructions,766,2317,37.161863
